# Processamento de Custos de Importacao - Questao 3

## Objetivo
Transformar o arquivo `custos_importacao.json`, que possui estrutura aninhada (dados historicos por produto), em um dataset tabular plano (flat), adequado para analises posteriores.

## Estrutura de Entrada
O JSON contem uma lista de produtos, cada um com:
- `product_id`: identificador do produto
- `product_name`: nome do produto
- `category`: categoria
- `historic_data`: lista de registros com `start_date` e `usd_price`

## Estrutura de Saida
CSV com colunas: `product_id`, `product_name`, `category`, `start_date`, `usd_price` (uma linha por entrada historica).

In [1]:
import pandas as pd
import json
import warnings

warnings.filterwarnings('ignore')

## 1. Carregamento e Exploracao do JSON

In [2]:
with open('../datasets/custos_importacao.json', 'r', encoding='utf-8') as f:
    dados_raw = json.load(f)

print(f"Total de produtos no JSON: {len(dados_raw)}")
print(f"Tipo do objeto raiz: {type(dados_raw).__name__}")
print(f"\nEstrutura do primeiro registro:")
print(json.dumps(dados_raw[0], indent=2, ensure_ascii=False))

Total de produtos no JSON: 150
Tipo do objeto raiz: list

Estrutura do primeiro registro:
{
  "product_id": 1,
  "product_name": "Transponder AIS Maré Magnum",
  "category": "eletrônicos",
  "historic_data": [
    {
      "start_date": "10/08/2016",
      "usd_price": 10583.63
    },
    {
      "start_date": "15/06/2018",
      "usd_price": 8778.36
    },
    {
      "start_date": "25/09/2018",
      "usd_price": 8023.87
    },
    {
      "start_date": "19/03/2019",
      "usd_price": 8772.78
    },
    {
      "start_date": "17/01/2020",
      "usd_price": 7918.18
    },
    {
      "start_date": "17/06/2020",
      "usd_price": 6310.01
    },
    {
      "start_date": "02/07/2021",
      "usd_price": 6586.7
    },
    {
      "start_date": "16/05/2022",
      "usd_price": 6538.2
    },
    {
      "start_date": "28/02/2023",
      "usd_price": 6360.91
    },
    {
      "start_date": "17/10/2023",
      "usd_price": 6574.8
    },
    {
      "start_date": "16/02/2024",
      "usd_p

In [3]:
qtd_historicos = [len(p['historic_data']) for p in dados_raw]

print(f"Entradas historicas por produto:")
print(f"  Minimo: {min(qtd_historicos)}")
print(f"  Maximo: {max(qtd_historicos)}")
print(f"  Media: {sum(qtd_historicos) / len(qtd_historicos):.1f}")
print(f"  Total de entradas: {sum(qtd_historicos)}")

Entradas historicas por produto:
  Minimo: 3
  Maximo: 15
  Media: 8.4
  Total de entradas: 1260


## 2. Achatamento (Flatten) da Estrutura Aninhada

Cada entrada de `historic_data` sera transformada em uma linha individual, preservando os atributos do produto pai.

In [4]:
registros = []
for produto in dados_raw:
    for entrada in produto['historic_data']:
        registros.append({
            'product_id': produto['product_id'],
            'product_name': produto['product_name'],
            'category': produto['category'],
            'start_date': entrada['start_date'],
            'usd_price': entrada['usd_price']
        })

df = pd.DataFrame(registros)

print(f"Registros gerados apos achatamento: {len(df)}")
print(f"Colunas: {list(df.columns)}")
print(f"\nPrimeiras linhas:")
df.head(10)

Registros gerados apos achatamento: 1260
Colunas: ['product_id', 'product_name', 'category', 'start_date', 'usd_price']

Primeiras linhas:


,product_id,product_name,category,start_date,usd_price
0,1,Transponder AIS Maré Magnum,eletrônicos,10/08/2016,10583.63
1,1,Transponder AIS Maré Magnum,eletrônicos,15/06/2018,8778.36
2,1,Transponder AIS Maré Magnum,eletrônicos,25/09/2018,8023.87
3,1,Transponder AIS Maré Magnum,eletrônicos,19/03/2019,8772.78
4,1,Transponder AIS Maré Magnum,eletrônicos,17/01/2020,7918.18
5,1,Transponder AIS Maré Magnum,eletrônicos,17/06/2020,6310.01
6,1,Transponder AIS Maré Magnum,eletrônicos,02/07/2021,6586.70
7,1,Transponder AIS Maré Magnum,eletrônicos,16/05/2022,6538.20
8,1,Transponder AIS Maré Magnum,eletrônicos,28/02/2023,6360.91
9,1,Transponder AIS Maré Magnum,eletrônicos,17/10/2023,6574.80


## 3. Conversao de Tipos

In [5]:
print("Tipos antes da conversao:")
print(df.dtypes)

Tipos antes da conversao:
product_id        int64
product_name     object
category         object
start_date       object
usd_price       float64
dtype: object


In [6]:
df['product_id'] = df['product_id'].astype(int)
df['start_date'] = pd.to_datetime(df['start_date'], format='%d/%m/%Y')
df['usd_price'] = df['usd_price'].astype(float)

print("Tipos apos conversao:")
print(df.dtypes)

Tipos apos conversao:
product_id               int64
product_name            object
category                object
start_date      datetime64[ns]
usd_price              float64
dtype: object


In [7]:
print(f"Intervalo de datas: {df['start_date'].min().strftime('%d/%m/%Y')} a {df['start_date'].max().strftime('%d/%m/%Y')}")
print(f"\nEstatisticas de preco (USD):")
print(df['usd_price'].describe())
print(f"\nDistribuicao por categoria:")
print(df['category'].value_counts())

Intervalo de datas: 04/01/2016 a 31/12/2025

Estatisticas de preco (USD):
count     1260.000000
mean      7890.234786
std       9961.166048
min         53.370000
25%        843.865000
50%       2991.195000
75%      11807.730000
max      48583.210000
Name: usd_price, dtype: float64

Distribuicao por categoria:
category
eletrônicos    436
ancoragem      413
propulsão      411
Name: count, dtype: int64


## 4. Validacao

In [8]:
print("=" * 60)
print("VALIDACAO DOS DADOS")
print("=" * 60)
print(f"Total de registros: {len(df)}")
print(f"Produtos distintos: {df['product_id'].nunique()}")
print(f"Valores nulos: {df.isnull().sum().sum()}")
print(f"\nVerificacao de tipos:")
print(f"  product_id - {df['product_id'].dtype} (esperado: int)")
print(f"  product_name - {df['product_name'].dtype} (esperado: object)")
print(f"  category - {df['category'].dtype} (esperado: object)")
print(f"  start_date - {df['start_date'].dtype} (esperado: datetime64)")
print(f"  usd_price - {df['usd_price'].dtype} (esperado: float64)")

VALIDACAO DOS DADOS
Total de registros: 1260
Produtos distintos: 150
Valores nulos: 0

Verificacao de tipos:
  product_id - int64 (esperado: int)
  product_name - object (esperado: object)
  category - object (esperado: object)
  start_date - datetime64[ns] (esperado: datetime64)
  usd_price - float64 (esperado: float64)


In [9]:
df = df.sort_values(['product_id', 'start_date']).reset_index(drop=True)

print("Amostra do dataset final (ordenado por product_id e start_date):")
df.head(15)

Amostra do dataset final (ordenado por product_id e start_date):


,product_id,product_name,category,start_date,usd_price
0,1,Transponder AIS Maré Magnum,eletrônicos,2016-08-10,10583.63
1,1,Transponder AIS Maré Magnum,eletrônicos,2018-06-15,8778.36
2,1,Transponder AIS Maré Magnum,eletrônicos,2018-09-25,8023.87
3,1,Transponder AIS Maré Magnum,eletrônicos,2019-03-19,8772.78
4,1,Transponder AIS Maré Magnum,eletrônicos,2020-01-17,7918.18
5,1,Transponder AIS Maré Magnum,eletrônicos,2020-06-17,6310.01
6,1,Transponder AIS Maré Magnum,eletrônicos,2021-07-02,6586.70
7,1,Transponder AIS Maré Magnum,eletrônicos,2022-05-16,6538.20
8,1,Transponder AIS Maré Magnum,eletrônicos,2023-02-28,6360.91
9,1,Transponder AIS Maré Magnum,eletrônicos,2023-10-17,6574.80


## 5. Exportacao para CSV

In [10]:
df.to_csv('../outputs/dados_processados/custos_importacao_flat.csv', index=False)
print("Arquivo salvo em: ../outputs/dados_processados/custos_importacao_flat.csv")
print(f"Registros exportados: {len(df)}")

Arquivo salvo em: ../outputs/dados_processados/custos_importacao_flat.csv
Registros exportados: 1260


## 6. Respostas as Validacoes

In [11]:
print("=" * 60)
print("RESPOSTAS - QUESTAO 3")
print("=" * 60)

print(f"\nQuestao 3.2 - Quantas entradas o dataset possui apos o achatamento?")
print(f"RESPOSTA: {len(df)} entradas")
print(f"\nDetalhamento:")
print(f"  Produtos: {df['product_id'].nunique()}")
print(f"  Total de entradas historicas: {len(df)}")

RESPOSTAS - QUESTAO 3

Questao 3.2 - Quantas entradas o dataset possui apos o achatamento?
RESPOSTA: 1260 entradas

Detalhamento:
  Produtos: 150
  Total de entradas historicas: 1260


## Resumo do Processamento

| Metrica | Valor |
|---|---|
| Produtos no JSON original | 150 |
| Total de entradas apos achatamento | 1260 |
| Colunas no CSV de saida | 5 |
| Valores nulos | 0 |